# Issue #120 — REAL Tripartite E2E Kaggle Verification

This notebook is designed for a fresh Kaggle session with **Internet enabled** and a **GPU enabled** when available. It checks out the exact production commit, installs the repository's actual package from `munajjam/pyproject.toml`, downloads only public artifacts, and runs the real segmenter, public sherpa-onnx Zipformer, canonical Quran reference provider, Wav2Vec2 provider, and `HybridQuranAligner`.

It never uses credentials, gated Quran-Lab artifacts, synthetic logits, fake boundaries, notebook-only model adapters, or `server.py`.

The notebook writes exactly one evidence artifact:

- `REAL_TRIPARTITE_E2E_EVIDENCE.json` only when the complete production result is genuinely verified.
- `REAL_TRIPARTITE_E2E_BLOCKER.json` when any real stage fails or the current production metadata reports that Zipformer canonical-target DP fusion is incomplete.

In [ ]:
# 1. Fresh Kaggle bootstrap and exact source verification
from pathlib import Path
import os, subprocess, sys, json, hashlib, platform, shutil

REPO_URL = "https://github.com/BXR7/HybridQuranAligner-.git"
BRANCH = "feature/issue-120-tripartite-hybrid"
EXPECTED_COMMIT = "f1f47f03190c5fd885de46eadaa853917a4c9fd5"
REPO = Path("/kaggle/working/HybridQuranAligner-")
PROJECT = REPO / "munajjam"
CACHE = Path("/kaggle/working/munajjam-model-cache")
CACHE.mkdir(parents=True, exist_ok=True)

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--force", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}"], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == EXPECTED_COMMIT, (actual_commit, EXPECTED_COMMIT)
assert PROJECT.joinpath("pyproject.toml").is_file(), "Expected the actual package project file under munajjam/"
assert not (REPO / "pyproject.toml").exists(), "The notebook must not assume a root pyproject.toml"
# Ensure no ambient Hugging Face credential is used by this notebook.
os.environ.pop("HF_TOKEN", None)
os.environ.pop("HF_HUB_TOKEN", None)
os.environ["MUNAJJAM_MODEL_CACHE"] = str(CACHE)
os.environ["PYTHONPATH"] = f"{PROJECT}:{os.environ.get('PYTHONPATH', '')}"
print("repository:", REPO_URL)
print("branch:", BRANCH)
print("commit:", actual_commit)
print("project:", PROJECT)

In [ ]:
# 2. Environment report and actual supported installation mechanism
import importlib.util

def run_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
if importlib.util.find_spec("torch") is None:
    print("torch is absent; Kaggle GPU bootstrap is required before this notebook.")
    raise RuntimeError("torch is required and was not preinstalled; refusing to silently install a different GPU stack")
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# This is the repository's real installation path. The repository has no root pyproject.toml.
run_install("-e", str(PROJECT))
# Additional runtime packages required by the actual pinned adapters. Do not reinstall torch.
run_install("sherpa-onnx>=1.13.8", "huggingface_hub", "recitations-segmenter==1.0.0", "transformers==4.51.3")
print("Installed the actual package from:", PROJECT / "pyproject.toml")

In [ ]:
# 3. Blocker/evidence helpers used by every later stage
from datetime import datetime, timezone

BLOCKER_PATH = Path("/kaggle/working/REAL_TRIPARTITE_E2E_BLOCKER.json")
EVIDENCE_PATH = Path("/kaggle/working/REAL_TRIPARTITE_E2E_EVIDENCE.json")
for stale in (BLOCKER_PATH, EVIDENCE_PATH):
    if stale.exists():
        stale.unlink()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_blocker(stage, exc, details=None):
    payload = {
        "status": "BLOCKED",
        "stage": stage,
        "exception": type(exc).__name__,
        "reason": str(exc),
        "details": details or {},
        "repository": REPO_URL,
        "branch": BRANCH,
        "commit": EXPECTED_COMMIT,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "suggested_next_action": "Install or make available the exact missing public runtime/model, then rerun this notebook; do not substitute synthetic output.",
    }
    BLOCKER_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    return payload

def fail(stage, exc, details=None):
    write_blocker(stage, exc, details)
    raise RuntimeError(f"REAL_TRIPARTITE_E2E blocked at {stage}: {exc}") from exc

In [ ]:
# 4. Run the repository's focused tests before downloading large model artifacts
focused = [
    "tests/unit/test_hybrid_aligner.py",
    "tests/unit/test_hybrid_reference.py",
    "tests/unit/test_zipformer_backend.py",
    "tests/unit/test_sherpa_zipformer_backend.py",
]
result = subprocess.run([sys.executable, "-m", "pytest", "-q", *focused], cwd=REPO, text=True)
print(result.stdout)
if result.returncode:
    raise RuntimeError("Focused Issue #120 tests failed")

In [ ]:
# 5. Acquire and verify the real Al-Fatiha 1:1 audio
# The expected sample contract is the one supplied with the Issue #120 evidence.
AUDIO_PATH = Path("/kaggle/working/001001_alafasy.mp3")
AUDIO_URL = "https://cdn.islamic.network/quran/audio/64/ar.alafasy/1.mp3"
EXPECTED_AUDIO_SHA256 = "b36a204a31ce2ac1e23ce3b4ed3e16f862f74cfa54de4b0a246fe3579620b802"
EXPECTED_AUDIO_SAMPLES = 96543
EXPECTED_AUDIO_RATE = 16000
EXPECTED_AUDIO_DURATION = 6.0339375

if not AUDIO_PATH.is_file():
    subprocess.run(["curl", "-L", "--fail", "--retry", "2", "--silent", AUDIO_URL, "-o", str(AUDIO_PATH)], check=True)
actual_audio_sha256 = sha256_file(AUDIO_PATH)
print("audio URL:", AUDIO_URL)
print("audio SHA-256:", actual_audio_sha256)
if actual_audio_sha256 != EXPECTED_AUDIO_SHA256:
    fail("audio_provenance", ValueError("downloaded audio does not match the required verified SHA-256"), {
        "expected_sha256": EXPECTED_AUDIO_SHA256,
        "actual_sha256": actual_audio_sha256,
        "source_url": AUDIO_URL,
    })

In [ ]:
# 6. Load real audio using the repository API and verify the decoded sample contract
from munajjam.hybrid_aligner import load_audio_file, AudioBuffer

audio = load_audio_file(AUDIO_PATH, sample_rate=EXPECTED_AUDIO_RATE)
print("sample rate:", audio.sample_rate)
print("samples:", len(audio.samples))
print("duration:", audio.duration)
if audio.sample_rate != EXPECTED_AUDIO_RATE or len(audio.samples) != EXPECTED_AUDIO_SAMPLES:
    fail("audio_decode", ValueError("decoded audio does not match the verified sample contract"), {
        "sample_rate": audio.sample_rate,
        "samples": len(audio.samples),
        "duration": audio.duration,
    })

In [ ]:
# 7. Real pinned recitation/breath segmenter smoke test
from munajjam.hybrid_aligner import QuranRecitationSegmenter, TransformersRecitationSegmenterBackend
from munajjam.hybrid_aligner.recitation_segmenter import RECITATION_SEGMENTER_REPOSITORY, RECITATION_SEGMENTER_REVISION

try:
    segmenter_backend = TransformersRecitationSegmenterBackend(device="cuda" if torch.cuda.is_available() else "cpu", cache_dir=str(CACHE))
    segmenter = QuranRecitationSegmenter(backend_factory=lambda: segmenter_backend)
    groups = segmenter.segment(audio)
except Exception as exc:
    fail("real_recitation_segmenter", exc, {
        "model": RECITATION_SEGMENTER_REPOSITORY,
        "revision": RECITATION_SEGMENTER_REVISION,
    })
print("segmenter:", RECITATION_SEGMENTER_REPOSITORY)
print("revision:", RECITATION_SEGMENTER_REVISION)
print("detected groups:", len(groups))
for i, group in enumerate(groups):
    print(i, "start=", group.start, "end=", group.end, "duration=", group.end - group.start, "score=", group.score)
if len(groups) != 1:
    fail("canonical_reference_assignment", ValueError("the supplied audio is expected to contain exactly one explicit 1:1 breath group"), {"group_count": len(groups)})

In [ ]:
# 8. Resolve and smoke-test the public Zipformer through the production Sherpa backend
from munajjam.hybrid_aligner import ModelManager, ModelSpec, SherpaZipformerBackend
from munajjam.hybrid_aligner.zipformer_backend import (
    PUBLIC_ZIPFORMER_MODEL, PUBLIC_ZIPFORMER_REPOSITORY, PUBLIC_ZIPFORMER_REVISION,
    PUBLIC_ZIPFORMER_TOKENS, ZIPFORMER_TOKEN_SHA256,
)

zipformer_spec = ModelSpec(
    repository=PUBLIC_ZIPFORMER_REPOSITORY,
    revision=PUBLIC_ZIPFORMER_REVISION,
    files=(PUBLIC_ZIPFORMER_TOKENS, PUBLIC_ZIPFORMER_MODEL),
    hashes={PUBLIC_ZIPFORMER_TOKENS: ZIPFORMER_TOKEN_SHA256},
)
try:
    zip_dir = ModelManager(CACHE).resolve(zipformer_spec, allow_download=True)
    token_hash = sha256_file(zip_dir / PUBLIC_ZIPFORMER_TOKENS)
    zip_model_hash = sha256_file(zip_dir / PUBLIC_ZIPFORMER_MODEL)
    zipformer_backend = SherpaZipformerBackend(zip_dir)
    zipformer_emissions = []
    for group in groups:
        emissions = zipformer_backend(audio, group)
        zipformer_emissions.append(emissions)
        print("group emissions:", len(emissions))
        print("first emissions:", emissions[:12])
except Exception as exc:
    fail("public_zipformer_sherpa", exc, {"repository": PUBLIC_ZIPFORMER_REPOSITORY, "revision": PUBLIC_ZIPFORMER_REVISION})
print("Zipformer repository:", PUBLIC_ZIPFORMER_REPOSITORY)
print("Zipformer revision:", PUBLIC_ZIPFORMER_REVISION)
print("tokens SHA-256:", token_hash)
print("ONNX SHA-256:", zip_model_hash)

In [ ]:
# 9. Canonical Quran reference and real Wav2Vec2 provider smoke test
from munajjam.hybrid_aligner import (
    CanonicalQuranReferenceProvider, CanonicalReferenceRequest,
    TransformersWav2Vec2LogitsProvider, Wav2Vec2ForcedAligner,
)
from munajjam.hybrid_aligner.wav2vec2_aligner import WAV2VEC2_REPOSITORY, WAV2VEC2_REVISION

reference_provider = CanonicalQuranReferenceProvider()
references = [CanonicalReferenceRequest(surah_id=1, ayah_start=1, ayah_end=1, riwaya="hafs")]
canonical = reference_provider.get_reference(references[0])
print("canonical text SHA-256:", canonical.text_sha256)
try:
    wav_provider = TransformersWav2Vec2LogitsProvider(
        device="cuda" if torch.cuda.is_available() else "cpu", cache_dir=str(CACHE)
    )
    wav_forced = Wav2Vec2ForcedAligner(logits_provider=wav_provider)
    token_ids, token_texts = wav_forced.encode_text(canonical.text)
    real_logits = wav_provider(audio.samples, audio.sample_rate)
    direct_spans = wav_forced.align_group(audio, groups[0], token_ids, token_texts)
except Exception as exc:
    fail("real_wav2vec2_forced_alignment", exc, {"repository": WAV2VEC2_REPOSITORY, "revision": WAV2VEC2_REVISION})
print("Wav2Vec2 repository:", WAV2VEC2_REPOSITORY)
print("Wav2Vec2 revision:", WAV2VEC2_REVISION)
print("device:", wav_provider._device)
print("vocabulary size:", wav_provider.model.config.vocab_size)
print("real logits shape:", real_logits.shape)
print("direct per-breath spans:", direct_spans)

In [ ]:
# 10. Complete current HybridQuranAligner orchestration with the real components
from munajjam.hybrid_aligner import HybridQuranAligner, ZipformerEvidence, ZipformerNeuralAligner

zip_evidence = ZipformerEvidence(
    repository=PUBLIC_ZIPFORMER_REPOSITORY,
    revision=PUBLIC_ZIPFORMER_REVISION,
    approved=True,
    vocabulary_size=251,
    blank_id=250,
    sample_rate=16000,
    feature_kind="kaldi-fbank",
    token_table_sha256=ZIPFORMER_TOKEN_SHA256,
)
reference_aligner = ZipformerNeuralAligner(
    zip_evidence, backend_factory=lambda: SherpaZipformerBackend(zip_dir)
)
# Reuse the real segmenter and real Wav2Vec2 provider already smoke-tested above.
pipeline = HybridQuranAligner(
    segmenter=segmenter,
    reference_aligner=reference_aligner,
    forced_aligner=wav_forced,
    canonical_reference_provider=reference_provider,
    require_canonical_references=True,
)
try:
    result = pipeline.align(audio, references=references)
except Exception as exc:
    fail("HybridQuranAligner", exc, {})
finally:
    pipeline.close()

print("final spans:", len(result.spans))
for span in result.spans:
    print(span)
print("warnings:", result.metadata.get("warnings"))
print("Zipformer status:", result.metadata.get("zipformer_reference_alignment_status"))

In [ ]:
# 11. Truthful final evidence gate: VERIFIED only if production metadata proves fusion
base = {
    "repository": REPO_URL,
    "branch": BRANCH,
    "commit": EXPECTED_COMMIT,
    "environment": {"python": sys.version, "platform": platform.platform()},
    "gpu": {"available": bool(torch.cuda.is_available()), "name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    "torch_version": torch.__version__,
    "audio": {"source_url": AUDIO_URL, "sha256": actual_audio_sha256, "sample_rate": audio.sample_rate, "samples": len(audio.samples), "duration": audio.duration},
    "segmenter": {"repository": RECITATION_SEGMENTER_REPOSITORY, "revision": RECITATION_SEGMENTER_REVISION, "groups": [g.__dict__ if hasattr(g, "__dict__") else {"start": g.start, "end": g.end, "score": g.score} for g in groups]},
    "zipformer": {"repository": PUBLIC_ZIPFORMER_REPOSITORY, "revision": PUBLIC_ZIPFORMER_REVISION, "model_sha256": zip_model_hash, "tokens_sha256": token_hash, "emissions": [[e.__dict__ if hasattr(e, "__dict__") else {"token_id": e.token_id, "start_frame": e.start_frame, "end_frame": e.end_frame, "score": e.score} for e in group] for group in zipformer_emissions]},
    "canonical_reference": {"text_sha256": canonical.text_sha256, "parts": [p.__dict__ if hasattr(p, "__dict__") else {"kind": p.kind, "source": p.source, "source_sha256": p.source_sha256} for p in canonical.parts]},
    "wav2vec2": {"repository": WAV2VEC2_REPOSITORY, "revision": WAV2VEC2_REVISION, "device": wav_provider._device, "vocabulary_size": int(wav_provider.model.config.vocab_size), "logits_shape": list(real_logits.shape), "direct_spans": [s.__dict__ if hasattr(s, "__dict__") else {"token": s.token, "start": s.start, "end": s.end, "score": s.score} for s in direct_spans]},
    "final_spans": [s.__dict__ if hasattr(s, "__dict__") else {"token": s.token, "start": s.start, "end": s.end, "score": s.score, "source": s.source} for s in result.spans],
    "warnings": result.metadata.get("warnings", []),
    "pipeline_metadata": result.metadata,
}

if result.metadata.get("zipformer_reference_alignment_completed") is True and result.metadata.get("zipformer_evidence_fused_into_final_spans") is True:
    evidence = {"status": "VERIFIED", **base}
    EVIDENCE_PATH.write_text(json.dumps(evidence, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    print("============================================================")
    print("FINAL REAL TRIPARTITE E2E RESULT: VERIFIED")
    print("============================================================")
    print(json.dumps(evidence, ensure_ascii=False, indent=2, default=str))
else:
    blocker = write_blocker(
        "canonical_zipformer_target_dp_fusion",
        RuntimeError("the current HybridQuranAligner metadata reports Zipformer emissions as unaligned and not fused into final spans"),
        {"pipeline_metadata": result.metadata, "real_stages_already_executed": ["audio", "segmenter", "public_zipformer_sherpa", "canonical_reference", "wav2vec2", "hybrid_orchestration"]},
    )
    print("============================================================")
    print("FINAL REAL TRIPARTITE E2E RESULT: BLOCKED")
    print("============================================================")
    print(json.dumps(blocker, ensure_ascii=False, indent=2))
    raise RuntimeError("Current commit does not yet provide canonical Zipformer target DP fusion")

In [ ]:
# 12. Run the public real-model integration test with the exact artifact/audio paths
# This cell is reached only when the full pipeline cell completes successfully.
env = os.environ.copy()
env.update({
    "RUN_REAL_MODEL": "1",
    "ZIPFORMER_ARTIFACT_DIR": str(zip_dir),
    "QURAN_AUDIO_PATH": str(AUDIO_PATH),
    "PYTHONPATH": f"{PROJECT}:{env.get('PYTHONPATH', '')}",
})
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/integration/test_zipformer_real_model.py"], cwd=REPO, env=env, check=True)